In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import export_text
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, accuracy_score,
    confusion_matrix, ConfusionMatrixDisplay, 
    precision_score, recall_score, f1_score
)
from sklearn.metrics.pairwise import cosine_similarity
import joblib

In [ ]:
URL = "http://localhost:8080/api/ml/dataset"

response = requests.get(URL)
response.raise_for_status()

data = response.json()
df = pd.DataFrame(data)

print("Real data shape:", df.shape)

Loading mock data and combining with real data

In [ ]:
with open("../data/mock_data.json", "r") as f:
    mock = json.load(f)

df_mock = pd.DataFrame(mock)

df_combined = pd.concat([df, df_mock], ignore_index=True)

print("Real rows:", len(df))
print("mock rows:", len(df_mock))
print("Combined rows:", len(df_combined))
print("\nLiked distribution:\n", df_combined["liked"].value_counts())


Cleaning the data:
   - text_columns --> contain the names of all the columns that contain words
   - fillna --> to fill N/A into cells that are empty, intead of leaving a blank gap
   - astype(str) --> make sure every value is treated as a string
   - df["liked"].astype(int) --> liked column marks wether user likes an artwork or not 
      - (1 liked, 0 not liked)
      
combining the columns:
   - to have the describtive features in one line rather than multipe columns, makes it easier to associate each feature with a specific artwork.

In [ ]:

text_columns = [
    "artist", "period", "culture", "medium",
    "preferredArtists", "preferredStyles",
    "preferredMediums", "preferredTimePeriods",
    "preferredMovements"
]

for col in text_columns:
    df_combined[col] = df_combined[col].fillna("").astype(str)

df_combined["liked"] = df_combined["liked"].astype(int)


# Combined Text Feature
df_combined["combined_text"] = (
    df_combined["artist"] + " " +
    df_combined["period"] + " " +
    df_combined["culture"] + " " +
    df_combined["medium"] + " " +
    df_combined["preferredArtists"] + " " +
    df_combined["preferredStyles"] + " " +
    df_combined["preferredMediums"] + " " +
    df_combined["preferredTimePeriods"] + " " +
    df_combined["preferredMovements"]
)

X_text = df_combined["combined_text"]
y = df_combined["liked"]

print("Total rows:", len(df_combined))
print("\nLiked distribution:\n", df_combined["liked"].value_counts())

# bar chart of liked distribution
df_combined["liked"].value_counts().sort_index().plot(kind="bar")
plt.xticks([0, 1], ["Not Liked (0)", "Liked (1)"], rotation=0)
plt.ylabel("Count")
plt.title("Liked/Not Liked Distribution")
plt.tight_layout()
plt.show()

splitting data 
- training data 80%
- testing data 20% 

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_text, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


TF-IDF Vectorization - turning words into numbers 
- max_features=500 keeps the 500 most informative words
- ngram_range=(1,2) captures single words AND two-word pairs like "oil painting"

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=500,
    ngram_range=(1, 2)
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

## Trainning models

Logistic regression:
- checks the features and finds patterns
- finds a single global boundary separating liked from not liked
- learns a weight for each of the 500 TF-IDF features

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

#testing the trained model
y_pred = model.predict(X_test_tfidf)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# confusion matrix
cm_lr = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_lr, display_labels=["Not Liked", "Liked"])
disp.plot()
plt.title("Logistic Regression Confusion Matrix")
plt.tight_layout()
plt.show()

- artwork_recommender :
    - trained model
- vectorizer: 
    - word to number translator 

In [ ]:
joblib.dump(model, "artwork_recommender.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")

KNN
- test accuracy of varius k numbers 
- pick the best one and use it  

In [ ]:

k_values = [3, 5, 7, 9, 11]
k_results = []
k_accuracies = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_tfidf, y_train)
    y_pred_k = knn.predict(X_test_tfidf)
    acc = accuracy_score(y_test, y_pred_k)
    k_results.append({"K": k, "Accuracy": acc})
    k_accuracies.append(acc)
    print(f"K={k} → Accuracy: {acc:.4f}")

# plot accuracy vs K
plt.figure(figsize=(6, 4))
plt.plot(k_values, k_accuracies, marker='o')
plt.xlabel("K (number of neighbors)")
plt.ylabel("Accuracy")
plt.title("KNN: Accuracy vs K")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# train the best K 
best_k = max(k_results, key=lambda x: x["Accuracy"])["K"]
print(f"\nBest K: {best_k}")

knn_model = KNeighborsClassifier(n_neighbors=best_k)
knn_model.fit(X_train_tfidf, y_train)

# evaluation
y_pred_knn = knn_model.predict(X_test_tfidf)

knn_accuracy = accuracy_score(y_test, y_pred_knn)
print("\nKNN Accuracy:", knn_accuracy)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_knn))

# save the model
joblib.dump(knn_model, "knn_recommender.pkl")

cm_knn = confusion_matrix(y_test, y_pred_knn)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_knn, display_labels=["Not Liked", "Liked"])
disp.plot()
plt.title(f"KNN Confusion Matrix (K={best_k})")
plt.tight_layout()
plt.show()

Decision tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# find the best max_depth
depth_values = [3, 5, 7, 9, None]
depth_results = []
depth_accuracies = []

for depth in depth_values:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train_tfidf, y_train)
    y_pred_d = dt.predict(X_test_tfidf)
    acc = accuracy_score(y_test, y_pred_d)
    depth_results.append({"max_depth": depth, "Accuracy": acc})
    depth_accuracies.append(acc)
    print(f"max_depth={depth} → Accuracy: {acc:.4f}")

# plot accuracy vs depth
depth_labels = [str(d) for d in depth_values]
plt.figure(figsize=(6, 4))
plt.plot(depth_labels, depth_accuracies, marker='o')
plt.xlabel("Tree Depth")
plt.ylabel("Accuracy")
plt.title("Decision Tree: Accuracy vs Depth")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# train the best max_depth
best_depth = max(depth_results, key=lambda x: x["Accuracy"])["max_depth"]
print(f"\nBest max_depth: {best_depth}")

dt_model = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
dt_model.fit(X_train_tfidf, y_train)

# evaluation
y_pred_dt = dt_model.predict(X_test_tfidf)

dt_accuracy = accuracy_score(y_test, y_pred_dt)
print("\nDecision Tree Accuracy:", dt_accuracy)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt))

# save the model
joblib.dump(dt_model, "dt_recommender.pkl")

# confusion matrix
cm_dt = confusion_matrix(y_test, y_pred_dt)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_dt, display_labels=["Not Liked", "Liked"])
disp.plot()
plt.title(f"Decision Tree Confusion Matrix (depth={best_depth})")
plt.tight_layout()
plt.show()


#decision rules
print("\nDecision Tree Rules:\n")
print(export_text(dt_model, feature_names=list(vectorizer.get_feature_names_out())))



Naive Bayes

In [ ]:

alpha_values = [0.1, 0.3, 0.5, 0.8, 1.0]
nb_results = []
nb_accuracies = []

for alpha in alpha_values:
    nb = MultinomialNB(alpha=alpha)
    nb.fit(X_train_tfidf, y_train)
    y_pred_nb = nb.predict(X_test_tfidf)
    acc = accuracy_score(y_test, y_pred_nb)
    nb_results.append({"Alpha": alpha, "Accuracy": acc})
    nb_accuracies.append(acc)
    print(f"Alpha={alpha} → Accuracy: {acc:.4f}")

# plot accuracy vs alpha
plt.figure(figsize=(6, 4))
plt.plot(alpha_values, nb_accuracies, marker='o')
plt.xlabel("Alpha (smoothing parameter)")
plt.ylabel("Accuracy")
plt.title("Naive Bayes: Accuracy vs Alpha")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# train best alpha
best_alpha = max(nb_results, key=lambda x: x["Accuracy"])["Alpha"]
print(f"\nBest Alpha: {best_alpha}")

nb_model = MultinomialNB(alpha=best_alpha)
nb_model.fit(X_train_tfidf, y_train)
y_pred_nb_final = nb_model.predict(X_test_tfidf)

print("\nNaive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb_final))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_nb_final))

# confusion matrix
cm_nb = confusion_matrix(y_test, y_pred_nb_final)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_nb, display_labels=["Not Liked", "Liked"])
disp.plot()
plt.title(f"Naive Bayes Confusion Matrix (alpha={best_alpha})")
plt.tight_layout()
plt.show()

joblib.dump(nb_model, "nb_recommender.pkl")

Cosine similarity

In [ ]:
# build a profile of what "liked" looks like from training data
liked_texts = df_combined.iloc[y_train[y_train == 1].index]["combined_text"]
liked_vectors = vectorizer.transform(liked_texts)
liked_profile = np.asarray(liked_vectors.mean(axis=0))

# score every test artwork against that profile
scores = cosine_similarity(liked_profile, X_test_tfidf)[0]

# predict liked=1 if score is above threshold
y_pred_cosine = (scores >= 0.1).astype(int)

print("Cosine Similarity Accuracy:", accuracy_score(y_test, y_pred_cosine))
print(classification_report(y_test, y_pred_cosine))

# confusion matrix
cm_cosine = confusion_matrix(y_test, y_pred_cosine)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_cosine, display_labels=["Not Liked", "Liked"])
disp.plot()
plt.title("Cosine Similarity Confusion Matrix")
plt.tight_layout()
plt.show()
#print confusion matrix values
print("Confusion Matrix:\n", cm_cosine)


Comparison 

In [ ]:

def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    return {
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_test, y_pred, zero_division=0), 4),
        "F1 Score": round(f1_score(y_test, y_pred, zero_division=0), 4),
    }

comparison_results = []
comparison_results.append(evaluate_model("Logistic Regression", model, X_test_tfidf, y_test))
comparison_results.append(evaluate_model(f"KNN (K={best_k})", knn_model, X_test_tfidf, y_test))
comparison_results.append(evaluate_model(f"Decision Tree (depth={best_depth})", dt_model, X_test_tfidf, y_test))
comparison_results.append(evaluate_model(f"Naive Bayes (alpha={best_alpha})", nb_model, X_test_tfidf, y_test))
comparison_results.append({
    "Model": "Cosine Similarity",
    "Accuracy": round(accuracy_score(y_test, y_pred_cosine), 4),
    "Precision": round(precision_score(y_test, y_pred_cosine, zero_division=0), 4),
    "Recall": round(recall_score(y_test, y_pred_cosine, zero_division=0), 4),
    "F1 Score": round(f1_score(y_test, y_pred_cosine, zero_division=0), 4),
})

comparison_df = pd.DataFrame(comparison_results)
print(comparison_df.to_string(index=False))

# bar chart comparing all models
plt.figure(figsize=(10, 5))
plt.bar(comparison_df["Model"], comparison_df["Accuracy"], color="steelblue")
plt.xlabel("Model")
plt.ylabel("Accuracy")
plt.title("Model Comparison: Accuracy")
plt.xticks(rotation=15, ha="right")
plt.ylim(0, 1)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()